# SarcasTone — Stage 1: Multimodal Feature Extraction & Drive Caching
### Optimized for Google Colab (T4 GPU - 16 GB VRAM)

[![Colab T4](https://img.shields.io/badge/Google_Colab-T4_GPU_Optimized-orange.svg?logo=googlecolab)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.1+-ee4c2c.svg?logo=pytorch)](https://pytorch.org/)
[![Transformers](https://img.shields.io/badge/🤗_Transformers-4.38+-yellow.svg)](https://huggingface.co/transformers/)

---

### Pipeline Overview
This notebook executes **Stage 1** of SarcasTone:
1. **Mounts Google Drive** for persistent storage of dataset, cache, and split manifests.
2. **Verifies T4 GPU** availability and CUDA environment (`torch.cuda.is_available()`).
3. **Installs required dependencies** from `requirements.txt`.
4. **Ingests annotations & creates deterministic stratified splits** (`splits.json`) with zero sample leakage.
5. **Extracts multimodal representations** across frozen pretrained backbones:
   * **Text:** `microsoft/deberta-v3-base` attention-masked mean pooling $\rightarrow \mathbb{R}^{768}$
   * **Speech Prosody:** 16-dimensional explicit vector ($F_0$ pitch dynamics, RMS energy, rhythm, pauses)
   * **Speech Acoustic:** `facebook/wav2vec2-base` temporal mean pooling $\rightarrow \mathbb{R}^{768}$
   * **Video:** `torchvision.models.resnet50` uniform 8-frame pooling $\rightarrow \mathbb{R}^{2048}$
6. **Persists extracted tensors** as `.pt` files to Google Drive with atomic manifest tracking for instant, idempotent resuming.

## 1. Mount Google Drive
Mount Google Drive to persist raw dataset media, offline extracted feature tensors, and deterministic split files across Colab runtimes.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted successfully at /content/drive")
except ImportError:
    print("ℹ️ Running outside Google Colab environment; skipping Drive mount.")

## 2. Check Hardware & T4 GPU Availability
Verify that CUDA is enabled and a Google Colab T4 GPU (or equivalent) is allocated.

In [ ]:
import sys
import torch

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available : {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"✓ GPU Allocated: {device_name}")
    print(f"✓ Total VRAM   : {vram_gb:.2f} GB")
    if "T4" in device_name:
        print("✓ Verified: NVIDIA Tesla T4 GPU ready for feature extraction!")
else:
    print("⚠️ WARNING: GPU is not enabled! To accelerate feature extraction:")
    print("   Select Runtime > Change runtime type > Hardware accelerator > T4 GPU")

## 3. Repository Setup & Dependency Installation
Navigate to or clone the SarcasTone repository and install the verified dependency stack.

In [ ]:
import os
import sys
from pathlib import Path

# Candidate repository locations
REPO_URL = "https://github.com/Tanmay-Shrivastav/SarcasTone.git"
REPO_DIR = Path("/content/SarcasTone")
DRIVE_REPO_DIR = Path("/content/drive/MyDrive/SarcasTone")

if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / "configs" / "default.yaml").exists():
    os.chdir(str(DRIVE_REPO_DIR))
    print(f"✓ Working directory set to Google Drive repository: {DRIVE_REPO_DIR}")
    if (DRIVE_REPO_DIR / ".git").exists():
        print("Checking for latest repository updates on Drive...")
        !git pull origin main || git pull
elif REPO_DIR.exists() and (REPO_DIR / "configs" / "default.yaml").exists():
    os.chdir(str(REPO_DIR))
    print(f"✓ Working directory set to: {REPO_DIR}")
    if (REPO_DIR / ".git").exists():
        print("Checking for latest repository updates...")
        !git pull origin main || git pull
else:
    print(f"Cloning repository from {REPO_URL} into /content/SarcasTone...")
    !git clone https://github.com/Tanmay-Shrivastav/SarcasTone.git /content/SarcasTone
    os.chdir("/content/SarcasTone")
    print("✓ Working directory set to: /content/SarcasTone")

# Add repository root to Python module search path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Install locked dependencies
!pip install -q -r requirements.txt
print("✓ Dependencies installed successfully.")

## 4. Configure Storage Paths
Load `configs/default.yaml` and initialize persistent Google Drive directories.

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = Path("configs/default.yaml")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Resolve paths
paths_cfg = cfg.get("paths", {})
data_root = Path(paths_cfg.get("data_root", "/content/drive/MyDrive/SarcasTone/data"))
cache_dir = Path(paths_cfg.get("cache_dir", "/content/drive/MyDrive/SarcasTone/cache"))
splits_file = Path(paths_cfg.get("splits_file", "/content/drive/MyDrive/SarcasTone/cache/splits.json"))
results_dir = Path(paths_cfg.get("results_dir", "/content/drive/MyDrive/SarcasTone/results"))

# Ensure directories exist
for p in [data_root, cache_dir, results_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("Configured Paths:")
print(f"  Data Root   : {data_root}")
print(f"  Cache Dir   : {cache_dir}")
print(f"  Splits File : {splits_file}")
print(f"  Results Dir : {results_dir}")

## 5. Dataset Ingestion & Deterministic Splitting
Ingest the MUStARD / MUStARD++ dataset structure:
- `data_root/annotations.json` (dialogue transcripts, speaker annotations, binary sarcasm labels)
- `data_root/utterances_final/<id>.mp4` (utterance audiovisual clips)
- `data_root/context_final/<id>_c.mp4` (preceding dialogue context clips)

`load_raw_samples` automatically resolves utterance video clips from `utterances_final/` and context clips from `context_final/`.
If dedicated `.wav` audio files are not present, `audio_path` automatically falls back to the `.mp4` video clip, and the 16 kHz audio track is extracted seamlessly in-memory during feature extraction.

In [ ]:
import json
import numpy as np
from src.data.dataset import (
    MultimodalSample,
    create_or_load_splits,
    load_raw_samples,
    verify_leakage,
)
from scripts.verify_splits import verify_splits

samples = None

# Check for real dataset annotations
if any((data_root / fname).is_file() for fname in ["annotations.json", "sarcasm_data.json", "mustard_plus.csv"]):
    print(f"Loading raw annotations from {data_root}...")
    samples = load_raw_samples(
        data_root=data_root,
        annotation_file=cfg["dataset"].get("annotation_file"),
        dataset_name=cfg["dataset"].get("name", "mustard"),
        utterance_dir=cfg["dataset"].get("utterance_dir", "utterances_final"),
        context_dir=cfg["dataset"].get("context_dir", "context_final"),
    )
else:
    print("ℹ️ No raw dataset annotations found in data_root.")
    print("Creating a 60-sample synthetic corpus for zero-download pipeline validation...")
    rng = np.random.RandomState(cfg.get("project", {}).get("seed", 42))
    shows = ["friends", "tbbt", "saty"]
    speakers = ["CHANDLER", "JOEY", "MONICA", "SHELDON", "LEONARD", "PENNY"]
    samples = {}
    for i in range(1, 61):
        uid = f"sample_{i:04d}"
        samples[uid] = MultimodalSample(
            utterance_id=uid,
            utterance_text=f"Utterance {i} test transcript for multimodal sarcasm modeling.",
            context_text=f"Preceding conversational turn context for utterance {i}.",
            speaker=speakers[i % len(speakers)],
            show=shows[i % len(shows)],
            label=int(rng.choice([0, 1])),
            audio_valid=False,
            video_valid=False,
        )

# Create or load deterministic splits
splits = create_or_load_splits(
    samples=samples,
    splits_path=splits_file,
    val_ratio=cfg["dataset"].get("val_ratio", 0.15),
    test_ratio=cfg["dataset"].get("test_ratio", 0.15),
    seed=cfg["project"].get("seed", 42),
    speaker_disjoint=cfg["dataset"].get("speaker_disjoint", False),
    overwrite=False,
)

print(f"✓ Split Counts: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

# Run strict split verification suite
split_ok = verify_splits(splits_path=splits_file, samples=samples)
assert split_ok, "Dataset split verification failed!"
print("✓ Strict zero-leakage verified across all splits.")

## 6. Initialize Pretrained Encoders on T4 GPU
Instantiate frozen feature extractors on the T4 GPU:
- `TextFeatureExtractor`: DeBERTa-v3 (`768,`)
- `ProsodyAcousticExtractor`: 16-dim prosodic statistics + Wav2Vec2 (`768,`)
- `VideoFeatureExtractor`: ResNet-50 (`2048,`)

In [ ]:
from src.features.text_features import TextFeatureExtractor
from src.features.prosody_features import ProsodyAcousticExtractor
from src.features.video_features import VideoFeatureExtractor

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = bool(cfg["training"].get("use_amp", True)) and device == "cuda"
print(f"Loading pretrained models onto device: {device} (AMP: {use_amp})...")

# 1. Text Encoder (DeBERTa-v3-base)
text_extractor = TextFeatureExtractor(
    model_name=cfg["encoders"].get("text_model", "microsoft/deberta-v3-base"),
    device=device,
    use_amp=use_amp,
)

# 2. Speech Acoustic & Prosody Extractor (Wav2Vec2 + Librosa)
speech_extractor = ProsodyAcousticExtractor(
    audio_model=cfg["encoders"].get("audio_model", "facebook/wav2vec2-base"),
    sample_rate=cfg["encoders"].get("sample_rate", 16000),
    max_audio_sec=cfg["encoders"].get("max_audio_sec", 15.0),
    device=device,
    use_amp=use_amp,
)

# 3. Video Frame Extractor (ResNet-50)
video_extractor = VideoFeatureExtractor(
    model_name="resnet50",
    num_frames=cfg["encoders"].get("num_video_frames", 8),
    frame_size=cfg["encoders"].get("frame_size", 224),
    device=device,
    use_amp=use_amp,
)

print("✓ All feature extractors loaded and initialized on GPU.")

## 7. Extract & Cache Features to Google Drive
Iterate through the samples, extract all modality tensors, and persist to `.pt` files.
Resuming is fully supported: if your runtime disconnects, re-running this cell skips already-cached samples.

In [ ]:
from tqdm.auto import tqdm
from src.features.cache_manager import FeatureCacheManager

cache_mgr = FeatureCacheManager(cache_dir=cache_dir, file_format="pt", overwrite=False)

all_sample_ids = list(samples.keys())
uncached_ids = cache_mgr.get_uncached_sample_ids(all_sample_ids, show_progress=True)

print(f"Total samples in dataset : {len(all_sample_ids)}")
print(f"Already cached on Drive  : {len(all_sample_ids) - len(uncached_ids)}")
print(f"Remaining to extract     : {len(uncached_ids)}")

# Extraction loop with throttled tqdm.auto progress bar (mininterval=1.0, miniters=4)
pbar = tqdm(
    uncached_ids,
    desc="Extracting & Caching Tensors",
    unit="sample",
    mininterval=1.0,
    miniters=4,
    leave=True,
)
for uid in pbar:
    sample = samples[uid]

    # 1. Text embedding (768,)
    text_emb = text_extractor.extract_single(
        text=sample.utterance_text,
        context=sample.context_text if cfg["dataset"].get("use_context", False) else None,
    )

    # 2. Speech prosody (16,) and acoustic embedding (768,)
    if sample.audio_path and sample.audio_valid:
        prosody_vec, audio_emb, audio_valid = speech_extractor.extract_from_file(sample.audio_path)
    else:
        prosody_vec = np.zeros(16, dtype=np.float32)
        audio_emb = np.zeros(768, dtype=np.float32)
        audio_valid = False

    # 3. Video frame embedding (2048,)
    if sample.video_path and sample.video_valid:
        video_emb, video_valid = video_extractor.extract_from_file(sample.video_path)
    else:
        video_emb = np.zeros(2048, dtype=np.float32)
        video_valid = False

    # Bundle and persist sample tensor
    features = {
        "text_emb": text_emb,
        "prosody_vec": prosody_vec,
        "audio_emb": audio_emb,
        "video_emb": video_emb,
        "label": sample.label,
        "audio_valid": audio_valid,
        "video_valid": video_valid,
        "metadata": {
            "speaker": sample.speaker,
            "show": sample.show,
        },
    }
    cache_mgr.save_sample_features(uid, features)
    pbar.set_postfix({"id": uid[:12], "label": sample.label}, refresh=False)

# Save manifest atomically
cache_mgr.save_manifest()
print("✓ Feature extraction and Drive caching successfully complete!")

## 8. Validate Cache & Inspect Extracted Tensors
Display diagnostic cache summary and load a sample tensor to verify dimensions.

In [ ]:
summary = cache_mgr.get_summary()

print("=" * 60)
print("FEATURE CACHE SUMMARY (GOOGLE DRIVE)")
print("=" * 60)
for k, v in summary.items():
    print(f"  {k:<22}: {v}")
print("=" * 60)

# Inspect first sample
sample_0 = all_sample_ids[0]
loaded = cache_mgr.load_sample_features(sample_0)

print(f"
Tensor Inspection for sample '{sample_0}':")
print(f"  text_emb    : shape {loaded['text_emb'].shape} (dtype: {loaded['text_emb'].dtype})")
print(f"  prosody_vec : shape {loaded['prosody_vec'].shape} (dtype: {loaded['prosody_vec'].dtype})")
print(f"  audio_emb   : shape {loaded['audio_emb'].shape} (dtype: {loaded['audio_emb'].dtype})")
print(f"  video_emb   : shape {loaded['video_emb'].shape} (dtype: {loaded['video_emb'].dtype})")
print(f"  label       : {loaded['label']}")
print(f"  audio_valid : {loaded['audio_valid']}")
print(f"  video_valid : {loaded['video_valid']}")

print("
" + "=" * 60)
print("STAGE 1 COMPLETE: All multimodal features are securely cached on Google Drive.")
print("Proceed to: notebooks/02_train_and_evaluate_colab.ipynb to run the 8-run ablation matrix.")
print("=" * 60)